In [17]:
import numpy as np

import troma
from troma import data_structure as ds
from troma import ConstraintSketch, ExplicitSketch
from troma import bind_optimizer, get_optimizer, optimize
from troma import matching_pursuit, get_matching_pursuit, bind_matching_pursuit
from troma import mcco_modeling
from troma import solve_via_mcco

In [9]:
#example
number_spins = 7

#abstract
spectrum_bin = [
    [0,0,0,0,0,1,0],
    [0,1,1,1,0,1,0],
    [0,0,1,1,1,1,0],
    [1,1,1,1,0,0,1]
]
spectrum_pos = [ds.dit_string_to_integer(s) for s in spectrum_bin]
spectrum_val = [-0.5,1.8,-0.3,1.5]

#explicit
full_spectrum = np.zeros((2**number_spins,))
full_spectrum[spectrum_pos] = spectrum_val


#Define marginals and sketch
interaction_size = 4
constraints = ConstraintSketch.build_nearest_neighbors_sketch(number_spins, interaction_size, 2)
y = ConstraintSketch.compute_marginal((spectrum_bin, spectrum_val), constraints)

In [10]:
#Abstract optimization

#NN spin chain (Default)
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size))

#Dual annealing
opti = get_optimizer("dual_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

#Simulated annealing
opti = get_optimizer("simulated_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

[[ 58.        2.175  ]
 [121.        1.40625]]
[[ 58.        2.175  ]
 [121.        1.40625]]
[[ 58.        2.175  ]
 [121.        1.40625]]


In [11]:
#Explicit methods

#Define marginals and sketch explicitly
sketch = ExplicitSketch.build_nearest_neighbors_sketch(number_spins,interaction_size)
y = ExplicitSketch.compute_marginal(full_spectrum, sketch)

#Brute force (Default)
print(matching_pursuit("explicit", y, sketch, 2))

sketch = ExplicitSketch.random_sketch(number_spins, 64)
y = ExplicitSketch.compute_marginal(full_spectrum, sketch)
print(matching_pursuit("explicit", y, sketch, 2))

[[ 58.        2.175  ]
 [121.        1.40625]]
[[ 58.           2.51348601]
 [121.           1.28652293]]


In [12]:
#example 2
number_spins = 6

#abstract
spectrum_bin = [
    [0,0,0,0,0,0],
    [0,1,0,1,0,1],
    [0,0,1,0,0,1],
    [1,1,1,1,1,1]
]
spectrum_pos = [ds.dit_string_to_integer(s) for s in spectrum_bin]
spectrum_val = [0.5,5,3,2]

#explicit
full_spectrum = np.zeros((2**number_spins,))
full_spectrum[spectrum_pos] = spectrum_val

#Define marginals and sketch
interaction_size = 2
constraints = ConstraintSketch.build_nearest_neighbors_sketch(number_spins, interaction_size, 2)
y = ConstraintSketch.compute_marginal((spectrum_bin, spectrum_val), constraints)

In [13]:
#Classical methods

#NN_spin_chain
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size))

#Dual annealing
opti = get_optimizer("dual_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

#Simulated annealing
opti = get_optimizer("simulated_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

[[21.    5.6 ]
 [ 9.    3.08]]
[[21.    5.6 ]
 [ 9.    3.08]]
[[21.   5.6]
 [15.   2.5]]


In [14]:
#Quantum methods

#Digital annealing
opti = get_optimizer("digital_annealing")
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

#QAOA
opti = bind_optimizer("qaoa", number_shots=4096)
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

[[21.    5.6 ]
 [ 9.    3.08]]
[[21.    5.6 ]
 [ 9.    3.08]]


In [ ]:
#Example 3
from problems_generator import compressible_opt_pb as co_pb

# number_spins = 12
# rules_reward = co_pb.generate_rules_and_rewards(2**number_spins, num_rules=3)

# number_spins = 12
# rules_reward = {(0, 0, 0, 0): 18, (1, 1, 0, 1): 5, (1, 1, 1, 1): 18}

number_spins = 16
rules_reward = {(0, 1, 1, 0): 2, (1, 1, 0, 1): 15, (0, 0, 1, 0): 1}
print(rules_reward)

{(0, 1, 1, 0): 2, (1, 1, 0, 1): 15, (0, 0, 1, 0): 1}


In [19]:
import matplotlib.pyplot as plt

full_spectrum = co_pb.compute_full_spectrum(number_spins, rules_reward)
print(np.where(full_spectrum == np.max(full_spectrum)))
# plt.bar(range(2**number_spins), full_spectrum)
# plt.show()

(array([56173]),)


In [23]:
from functools import partial

objective = partial(co_pb.estimate_cost, rules_rewards=rules_reward)

result = solve_via_mcco(
    objective_function=objective,
    number_samples=5000,
    dit_string_length=number_spins,
    interaction_size=4,
    iteration_number=5,
    thereshold_parameter="Auto",
    optimizer=get_optimizer("spin_chain_nn_max"),
)

print("spectrum_pos:", result["spectrum_pos"])
print("spectrum_val:", result["spectrum_val"])
print("spectrum_bin:", result["spectrum_bin"])
print("solution:\n", result["solution"])

spectrum_pos: (218, 436, 879, 950, 1709, 1758, 1772, 2486, 2781, 2933, 2941, 3034, 3293, 3436, 3482, 3518, 4333, 4534, 4969, 4972, 4981, 4982, 5549, 5847, 5885, 6363, 6581, 6586, 6733, 6766, 6767, 6964, 6965, 6970, 6982, 6987, 7008, 7017, 7023, 7066, 7084, 7094, 7098, 7117, 7588, 7645, 8621, 9053, 9869, 9908, 9909, 9940, 9944, 10678, 10683, 11191, 11226, 11547, 11611, 11624, 11689, 11693, 11709, 11725, 11730, 11753, 11767, 11771, 11988, 12012, 12139, 12214, 12251, 12731, 13019, 13165, 13166, 13175, 13416, 13417, 13494, 13526, 13530, 13534, 13709, 13732, 13733, 13930, 13956, 13962, 13965, 13993, 13994, 14019, 14054, 14061, 14066, 14072, 14077, 14106, 14183, 14245, 14298, 14925, 15069, 15117, 15179, 15195, 15322, 15578, 15803, 15834, 15853, 16088, 16090, 16093, 16821, 17114, 17243, 18102, 18134, 18861, 19311, 19383, 19725, 19739, 19757, 19867, 19876, 19881, 19897, 19958, 20205, 20909, 22157, 22234, 22971, 23021, 23053, 23259, 23380, 23391, 23406, 23460, 23472, 23474, 23862, 25019, 25527,

In [62]:

# Benchmark : sur N problèmes générés avec co_pb.generate_rules_and_rewards (taille 2**16),
# combien de fois solve_via_mcco retrouve-t-il le maximum global ?

import time

number_spins_bench = 16
n_problems = 100
num_rules = 3
success_count = 0
timings = []

for i in range(n_problems):
    # Générer un problème aléatoire
    rules_reward_bench = co_pb.generate_rules_and_rewards(2**number_spins_bench, num_rules=num_rules)

    # Calculer le vrai maximum
    full_spectrum_bench = co_pb.compute_full_spectrum(number_spins_bench, rules_reward_bench)
    true_max_positions = set(np.where(full_spectrum_bench == np.max(full_spectrum_bench))[0].tolist())

    # Résoudre via MCCO
    objective_bench = partial(co_pb.estimate_cost, rules_rewards=rules_reward_bench)
    t0 = time.time()
    result_bench = solve_via_mcco(
        objective_function=objective_bench,
        number_samples=5000,
        dit_string_length=number_spins_bench,
        interaction_size=4,
        iteration_number=5,
        thereshold_parameter="Auto",
        optimizer=get_optimizer("spin_chain_nn_max"),
    )
    elapsed = time.time() - t0
    timings.append(elapsed)

    # Vérifier si le maximum est trouvé dans la 1ère colonne de solution
    found_positions = set(result_bench["solution"][:, 0].tolist())
    hit = bool(true_max_positions & found_positions)
    success_count += int(hit)

    print(f"[{i+1:02d}/{n_problems}] {'✓ HIT' if hit else '✗ MISS'} | {elapsed:.1f}s")

print(f"\n=== Résultats benchmark ===")
print(f"Taille du problème : 2^{number_spins_bench} = {2**number_spins_bench}")
print(f"Succès : {success_count}/{n_problems}  ({100*success_count/n_problems:.0f}%)")
print(f"Temps moyen par problème : {np.mean(timings):.2f}s  (total : {np.sum(timings):.1f}s)")


[01/100] ✗ MISS | 0.6s
[02/100] ✓ HIT | 0.6s
[03/100] ✓ HIT | 0.6s
[04/100] ✓ HIT | 0.6s
[05/100] ✗ MISS | 0.6s
[06/100] ✓ HIT | 0.6s
[07/100] ✓ HIT | 0.6s
[08/100] ✓ HIT | 0.6s
[09/100] ✓ HIT | 0.6s
[10/100] ✓ HIT | 0.6s
[11/100] ✓ HIT | 0.6s
[12/100] ✓ HIT | 0.6s
[13/100] ✓ HIT | 0.6s
[14/100] ✓ HIT | 0.6s
[15/100] ✓ HIT | 0.6s
[16/100] ✓ HIT | 0.6s
[17/100] ✓ HIT | 0.8s
[18/100] ✗ MISS | 0.6s
[19/100] ✓ HIT | 0.6s
[20/100] ✗ MISS | 0.6s
[21/100] ✓ HIT | 0.6s
[22/100] ✓ HIT | 0.6s
[23/100] ✓ HIT | 0.6s
[24/100] ✓ HIT | 0.6s
[25/100] ✓ HIT | 0.6s
[26/100] ✓ HIT | 0.6s
[27/100] ✓ HIT | 0.6s
[28/100] ✓ HIT | 0.6s
[29/100] ✗ MISS | 0.6s
[30/100] ✗ MISS | 0.6s
[31/100] ✓ HIT | 0.6s
[32/100] ✗ MISS | 0.8s
[33/100] ✓ HIT | 0.6s
[34/100] ✗ MISS | 0.7s
[35/100] ✓ HIT | 0.6s
[36/100] ✓ HIT | 0.6s
[37/100] ✓ HIT | 0.6s
[38/100] ✓ HIT | 0.6s
[39/100] ✗ MISS | 0.6s
[40/100] ✓ HIT | 0.6s
[41/100] ✓ HIT | 0.6s
[42/100] ✓ HIT | 0.6s
[43/100] ✓ HIT | 0.6s
[44/100] ✗ MISS | 0.7s
[45/100] ✗ MISS | 0.6s